# 03 · Modelos

Ordinal, riesgo discreto, gradient boosting y Transformer; calibración en el 20 % por vehículo.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('..')
import gc
import pandas as pd
import utils

ID, T = utils.ID, utils.T

In [ ]:
points = {
    'train': pd.concat([utils.load_processed('train_labels_conv'),
                        utils.load_processed('train_labels_landmark')])[[ID, T]],
    'validation': utils.load_processed('validation_eval')[[ID, T]],
    'test': utils.load_processed('test_eval')[[ID, T]],
}

for split, pts in points.items():
    ro = utils.load_processed(f'{split}_readouts')
    F = utils.build_features(ro, pts)
    del ro; gc.collect()

    spec = utils.load_processed(f'{split}_specifications')
    spec = spec.astype({c: 'category' for c in spec.columns if c.startswith('Spec_')})
    F = F.merge(spec, on=ID, how='left')

    utils.to_parquet(F, f'{split}_features')
    print(split, F.shape, '| puntos pedidos:', len(pts.drop_duplicates()))
    del F; gc.collect()

In [ ]:
import numpy as np

F    = utils.load_processed('train_features')
conv = utils.load_processed('train_labels_conv')
lm   = utils.load_processed('train_labels_landmark')

data = {
    'convencional': F.merge(conv[[ID, T, 'y_conv']], on=[ID, T]).rename(columns={'y_conv': 'y'}),
    'landmarking':  F.merge(lm.loc[lm['y_cens'].notna(), [ID, T, 'y_cens']], on=[ID, T])
                     .rename(columns={'y_cens': 'y'}),
}
for d in data.values():
    d['y'] = d['y'].astype(int)

cal_ids = utils.calibration_split(F[ID])          # 20 % de vehículos para calibración
feats = [c for c in F.columns if c != ID]          # time_step se queda: es la edad

Fv = utils.load_processed('validation_features').merge(
        utils.load_processed('validation_eval')[[ID, T, 'class_label']], on=[ID, T])
Xv, yv = Fv[feats], Fv['class_label'].to_numpy()

for name, d in data.items():
    fit = d[~d[ID].isin(cal_ids)]
    print(f'{name}: {len(fit)} filas de entrenamiento, {len(d) - len(fit)} de calibración')

In [ ]:
models, res = {}, []

for name, d in data.items():
    fit = d[~d[ID].isin(cal_ids)]
    b = utils.train_lgbm(fit[feats], fit['y'], Xv, yv)
    models[name] = b
    P = b.predict(Xv, num_iteration=b.best_iteration)
    for rule, decide in [('argmax', utils.decide_argmax), ('min_costo', utils.decide_min_cost)]:
        yhat = decide(P)
        res.append({'formulación': name, 'regla': rule,
                    'costo_total': utils.total_cost(yv, yhat),
                    '% a revisión': round(100 * (yhat > 0).mean(), 1),
                    'iteraciones': b.best_iteration})

for c in [0, 4]:
    yhat = np.full(len(yv), c)
    res.append({'formulación': 'trivial', 'regla': f'siempre {c}',
                'costo_total': utils.total_cost(yv, yhat),
                '% a revisión': 100.0 * (c > 0)})

pd.DataFrame(res)

In [ ]:
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# --- 1. ¿Mejora el COSTO con más árboles aunque el log-loss empeore?
d = data['convencional']
fit = d[~d[ID].isin(cal_ids)]
p = dict(objective='multiclass', num_class=5, learning_rate=0.05, num_leaves=31,
         min_data_in_leaf=50, feature_fraction=0.5, bagging_fraction=0.8,
         bagging_freq=1, seed=0, verbose=-1)
b = lgb.train(p, lgb.Dataset(fit[feats], fit['y']), num_boost_round=300)

print('árboles | costo val (min_costo) | % revisión | AUC falla vs sana')
for k in [2, 10, 25, 50, 100, 200, 300]:
    P = b.predict(Xv, num_iteration=k)
    yhat = utils.decide_min_cost(P)
    auc = roc_auc_score(yv > 0, 1 - P[:, 0])
    print(f'{k:7d} | {utils.total_cost(yv, yhat):21d} | {100*(yhat>0).mean():9.1f} | {auc:.3f}')

# --- 2. ¿Se distinguen los cortes de train de los puntos de validation?
adv = pd.concat([fit[feats].assign(es_val=0), Xv.assign(es_val=1)], ignore_index=True)
adv = adv.astype({c: 'category' for c in feats if c.startswith('Spec_')})
idx = np.random.default_rng(0).permutation(len(adv))
tr_i, te_i = idx[: len(idx) // 2], idx[len(idx) // 2:]
a = lgb.train(dict(objective='binary', verbose=-1, seed=0),
              lgb.Dataset(adv.loc[tr_i, feats], adv.loc[tr_i, 'es_val']), num_boost_round=200)
print('\nAUC train vs validation:', round(roc_auc_score(adv.loc[te_i, 'es_val'],
                                                         a.predict(adv.loc[te_i, feats])), 3))
imp = pd.Series(a.feature_importance('gain'), index=feats).sort_values(ascending=False)
print('Variables que más los distinguen:\n', imp.head(10).round(0).to_string())

In [ ]:
cols = ['272_0', '272_3', '272_4']

print('=== 1. CSV crudo (acumulados) ===')
for split in ['train', 'validation']:
    raw = utils.load_raw(split, 'operational_readouts')
    print(f'\n{split}:')
    print(raw[cols].describe().round(1).to_string())
    print('  % de ceros:', (raw[cols] == 0).mean().mul(100).round(1).to_dict())
    del raw

print('\n=== 2. Tasas d_ (01_datos) ===')
for split in ['train', 'validation']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=[f'd_{c}' for c in cols] + ['dt'])
    print(f'\n{split}:')
    print(ro.describe().round(3).to_string())
    del ro

print('\n=== 3. Resumen de los cortes (03_modelos) ===')
comp = pd.DataFrame({'train': fit['d_272_0_mean'].describe(), 'validation': Xv['d_272_0_mean'].describe()})
print(comp.round(3).to_string())

Ese AUC de 1 es preocupante tengo que revisar si hay fuga de información entre el train y el validation.  El modelo distingue con precisión perfecta si una fila viene de train o de validation, y lo hace casi solo con una variable: d_272_0_mean. Su importancia es 8 veces la de la siguiente.

In [ ]:
cols = ['272_0', '272_3', '272_4']

print('=== 1. CSV crudo (acumulados) ===')
for split in ['train', 'validation']:
    raw = utils.load_raw(split, 'operational_readouts')
    print(f'\n{split}:')
    print(raw[cols].describe().round(1).to_string())
    print('  % de ceros:', (raw[cols] == 0).mean().mul(100).round(1).to_dict())
    del raw

print('\n=== 2. Tasas d_ (01_datos) ===')
for split in ['train', 'validation']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=[f'd_{c}' for c in cols] + ['dt'])
    print(f'\n{split}:')
    print(ro.describe().round(3).to_string())
    del ro

print('\n=== 3. Resumen de los cortes (03_modelos) ===')
comp = pd.DataFrame({'train': fit['d_272_0_mean'].describe(), 'validation': Xv['d_272_0_mean'].describe()})
print(comp.round(3).to_string())

La diferencia viene del dataset, no de nuestro código. En el CSV crudo, la variable 272 de validation es unas 7,5 veces más pequeña que la de train: la media es 9,1 millones en train y 1,2 millones en validation, y las medianas son 7,0 millones contra 0,9 millones. Nuestro procesamiento solo arrastra esa diferencia hasta las tasas d_ y el resumen. Y dt es casi idéntico en ambos, así que el muestreo en el tiempo no es la causa.

Que sea un factor tan grande y tan parejo en todos los cuantiles sugiere algo de registro o de escala en el dataset, no un cambio real en cómo se usan los camiones. Antes de decidir qué hacer hay que responder dos preguntas:

- ¿Pasa solo con la 272 o también con otras variables?
- ¿Test se parece a validation o a train?

In [ ]:
d_cols = [c for c in pd.read_parquet(utils.PROCESSED / 'train_readouts.parquet', columns=None).columns
          if c.startswith('d_')] if False else None  # (no cargar todo dos veces)

import pyarrow.parquet as pq
d_cols = [c for c in pq.read_schema(utils.PROCESSED / 'train_readouts.parquet').names if c.startswith('d_')]

med = {}
for split in ['train', 'validation', 'test']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=d_cols)
    med[split] = ro.median()
    del ro

med = pd.DataFrame(med)
med['val/train'] = med['validation'] / med['train']
med['test/train'] = med['test'] / med['train']
med['var'] = med.index.str.extract(r'd_(\d+)_', expand=False)

print('Razón de medianas por variable (1 = igual a train):')
print(med.groupby('var')[['val/train', 'test/train']].median().round(2).to_string())

print('\nTasas d_ con razón fuera de [0.5, 2]:')
out = med[(med['val/train'] - 1).abs().gt(0.5) | (med['test/train'] < 0.5) | (med['test/train'] > 2)]
print(out[['train', 'validation', 'test', 'val/train', 'test/train']].round(2).to_string())